In [ ]:
#Test Data Generation when yFinance was still working
import yfinance as yf
import pandas as pd
import requests
import csv
import json
import pyarrow

tickers = []


stock_data = []
with open('static/stock_info.csv') as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        tickers.append(row[1])
for ticker in tickers:
    try:
        data = yf.download(ticker, period = '1y', threads=False, interval = '1mo', auto_adjust=False)
        data = data.reset_index()
        data['Date']= data['Date'].astype(str)
        data['Symbol'] = ticker
        data = data.dropna()
        stock_data.append(data)
    except:
        print("Error")

combined_data = pd.concat(stock_data)

combined_data.to_parquet("static/stock_data.parquet", engine = "pyarrow")    

In [7]:
import csv
file_path = 'static/trial_stock.csv'

def data_call():
    tickers = []
    with open(file_path, 'r') as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            tickers.append(row[0])
    print(tickers)
data_call()

['AAPL', 'QQQ', 'IXIC', 'VFIAX', 'TRP', 'SVI', 'DQ7A', 'VOW3']


In [2]:
import websocket
import json
import os
import threading

api_key = "ff3bacef82be48e983515f69a6c42a0e"
print("API Key:", api_key)
if not api_key:
    print("❌ ERROR: TWELVEDATA_API_KEY not set")
    exit()

def on_open(ws):
    print("✅ WebSocket connected")
    msg = {
        "action": "subscribe",
        "params": {
            "symbols": "BTC/USD",
            "apikey": api_key
        }
    }
    ws.send(json.dumps(msg))

def on_message(ws, message):
    print("📩 Message:", message)

def on_error(ws, error):
    print("❌ Error:", error)

def on_close(ws, code, reason):
    print(f"🔌 Closed: {code} — {reason}")

def setup_ws():
    ws = websocket.WebSocketApp(
        f"wss://ws.twelvedata.com/v1/quotes?apikey={api_key}",
        on_open=on_open,
        on_message=on_message,
        on_error=on_error,
        on_close=on_close
    )
    ws.run_forever()

setup_ws()


API Key: ff3bacef82be48e983515f69a6c42a0e
❌ Error: Handshake status 404 Not Found -+-+- {'date': 'Mon, 24 Mar 2025 04:12:33 GMT', 'content-type': 'text/plain', 'content-length': '18', 'connection': 'keep-alive', 'access-control-allow-credentials': 'true', 'access-control-allow-headers': 'Content-Type, Accept-Encoding, Authorization, Accept, Origin, X-Requested-With', 'access-control-allow-methods': 'GET, OPTIONS', 'access-control-allow-origin': '*', 'cf-cache-status': 'DYNAMIC', 'report-to': '{"endpoints":[{"url":"https:\\/\\/a.nel.cloudflare.com\\/report\\/v4?s=kotpmha8vS9GW2jVX4qFZKs3bvN%2BIFjTw1KPiZQ%2F4Tunr2guMC6mle3n216iqPAbdTFVnOFNS8Us%2B9u99VB2zXxPzmDpjXI8v5i%2F2q3%2BMWsSEPWdQmRIy3dxZ26FI1XqEj5b"}],"group":"cf-nel","max_age":604800}', 'nel': '{"success_fraction":0,"report_to":"cf-nel","max_age":604800}', 'server': 'cloudflare', 'cf-ray': '925354b54fe249aa-EWR', 'server-timing': 'cfL4;desc="?proto=TCP&rtt=5265&min_rtt=4478&rtt_var=1962&sent=5&recv=7&lost=0&retrans=0&sent_bytes=28